# FORECASTING

## 1. Paths & Setup

In [22]:
import joblib
import numpy as np
import pandas as pd

In [ ]:
from collections import defaultdict, deque

BASE      = "C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/"
MODEL_DIR = "C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Model/"

PATH_TEST_ORIG   = BASE + "test.csv"
PATH_TEST_FE     = BASE + "df_test_fe.csv"
PATH_TRAIN_CLEAN = BASE + "df_train_clean.csv"
PATH_X_TRAIN     = BASE + "X_train.csv"
PATH_MODEL       = MODEL_DIR + "gb_model.pkl"
PATH_OUTPUT      = BASE + "sample submission.csv"

model = joblib.load(PATH_MODEL)

Model loaded: RegressorChain


## 2. Recover Feature Column Order

In [24]:
# The model was trained after dropping ['team', 'opponent', 'Id', 'match_id'].
# We load X_train headers to get the exact column order expected by the model.
X_train_header = pd.read_csv(PATH_X_TRAIN, nrows=0)
FEATURE_COLS = [c for c in X_train_header.columns
                if c not in ['team', 'opponent', 'Id', 'match_id']]

print(f"Feature columns : {len(FEATURE_COLS)}")
print(FEATURE_COLS)

Feature columns : 82
['gender', 'is_home', 'neutral', 'tournament', 'venue_country', 'days_since_last_match_team', 'days_since_last_match_opp', 'elo_team', 'elo_opponent', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'first_match_team', 'first_match_opp', 'team_points_last5', 'team_points_last10', 'team_gd_last5', 'team_avg_goals_last5', 'team_avg_conceded_last5', 'team_win_rate_last10', 'h2h_points_last5', 'h2h_gd_last5', 'opp_points_last5', 'opp_points_last10', 'opp_gd_last5', 'opp_avg_goals_last5', 'opp_avg_conceded_last5', 'opp_win_rate_last10', 'points_last5_diff', 'gd_last5_diff', 'tournament_tier', 'year', 'month', 'is_recent', 'is_very_recent', 'season_num', 'elo_diff', 'elo_ratio', 'attack_vs_def_team', 'attack_vs_def_opp', 'xg_advantage', 'momentum_diff', 'momentum_ratio', 'pts_diff_last10', 'team_attack_consistency', 'opp_attack_consistency', 'attack_con

## 3. Load Files for Forecasting

In [ ]:
# Raw test file: needed for match dates (date was dropped in FE)
df_test_orig = pd.read_csv(PATH_TEST_ORIG, parse_dates=['date'])
if 'id' in df_test_orig.columns:                     # normalise lowercase variant
    df_test_orig.rename(columns={'id': 'Id'}, inplace=True)

# Feature-engineered test (keep match_id for pairing, then re-attach date)
df_test_fe = pd.read_csv(PATH_TEST_FE)
df_test_fe = df_test_fe.sort_values('match_id').reset_index(drop=True)

date_map = df_test_orig.set_index('Id')['date']
df_test_fe['date'] = pd.to_datetime(df_test_fe['Id'].map(date_map))

# Training data for history seed
df_train_clean = pd.read_csv(PATH_TRAIN_CLEAN, parse_dates=['date'])
df_train_clean = df_train_clean.sort_values('date').reset_index(drop=True)

print(f"Test matches : {df_test_fe['match_id'].nunique()}")
print(f"Test rows    : {len(df_test_fe)}")
print(f"Train rows   : {len(df_train_clean)}")

Test matches : 21211
Test rows    : 42422
Train rows   : 78772


## 4. Initialise Team History from Training Data

In [ ]:
# team_history[team]       → deque(maxlen=10): last ≤10 match result dicts
# team_last_date[team]     → pd.Timestamp of most recent match
# h2h_history[(team,opp)]  → deque(maxlen=5):  last ≤5 h2h result dicts

team_history   = defaultdict(lambda: deque(maxlen=10))
team_last_date = {}
h2h_history    = defaultdict(lambda: deque(maxlen=5))


def _add_one_perspective(team, opp, goals_for, goals_against, match_date):
    win  = int(goals_for > goals_against)
    draw = int(goals_for == goals_against)
    pts  = 3 * win + draw
    gd   = goals_for - goals_against

    team_history[team].append({
        'goals_for': goals_for, 'goals_against': goals_against,
        'points': pts, 'gd': gd, 'win': win
    })
    prev = team_last_date.get(team, pd.NaT)
    if pd.isna(prev) or match_date > prev:
        team_last_date[team] = match_date

    h2h_history[(team, opp)].append({'points': pts, 'gd': gd})


def update_history(team, opp, team_goals, opp_goals, match_date):
    """Register a completed match (both perspectives) in the history state."""
    _add_one_perspective(team, opp, team_goals, opp_goals, match_date)
    _add_one_perspective(opp, team, opp_goals, team_goals, match_date)


# ── Seed from training data ────────────────────────────────────────────────
if 'match_key' not in df_train_clean.columns:
    pair_key = df_train_clean[['team', 'opponent']].apply(
        lambda x: '||'.join(sorted([str(x['team']), str(x['opponent'])])), axis=1)
    df_train_clean['match_key'] = df_train_clean['date'].astype(str) + '||' + pair_key

train_matches = (df_train_clean
                 .dropna(subset=['team_goals', 'opp_goals'])
                 .drop_duplicates('match_key')
                 .sort_values('date'))

for _, row in train_matches.iterrows():
    update_history(row['team'], row['opponent'],
                   row['team_goals'], row['opp_goals'], row['date'])

print(f"History seeded: {len(team_history)} teams | {len(h2h_history)} h2h pairs")

History seeded: 287 teams | 11866 h2h pairs


## 5. Feature Computation Helpers

In [ ]:
def compute_ts_features(team, opp, match_date):
    """
    Compute all time-series features for a match using the current
    state of team_history / h2h_history (i.e. after all earlier matches).
    """
    # days_since_last_match
    prev_t = team_last_date.get(team, pd.NaT)
    prev_o = team_last_date.get(opp,  pd.NaT)

    dslm_team  = int((match_date - prev_t).days) if not pd.isna(prev_t) else 0
    dslm_opp   = int((match_date - prev_o).days) if not pd.isna(prev_o) else 0
    first_team = int(pd.isna(prev_t) or len(team_history[team]) == 0)
    first_opp  = int(pd.isna(prev_o) or len(team_history[opp])  == 0)

    # Team rolling stats
    th = list(team_history[team])
    if not th:
        team_pts5 = team_pts10 = team_gd5 = 0
        team_avg_goals5 = team_avg_conceded5 = 0.0
        team_wr10 = 0.5
    else:
        t5  = th[-5:]          # last ≤5  matches
        t10 = th               # last ≤10 matches (deque maxlen=10)
        team_pts5          = sum(m['points']       for m in t5)
        team_pts10         = sum(m['points']       for m in t10)
        team_gd5           = sum(m['gd']           for m in t5)
        team_avg_goals5    = float(np.mean([m['goals_for']     for m in t5]))
        team_avg_conceded5 = float(np.mean([m['goals_against'] for m in t5]))
        team_wr10          = float(np.mean([m['win']           for m in t10]))

    # Opponent rolling stats
    oh = list(team_history[opp])
    if not oh:
        opp_pts5 = opp_pts10 = opp_gd5 = 0
        opp_avg_goals5 = opp_avg_conceded5 = 0.0
        opp_wr10 = 0.5
    else:
        o5  = oh[-5:]
        o10 = oh
        opp_pts5          = sum(m['points']       for m in o5)
        opp_pts10         = sum(m['points']       for m in o10)
        opp_gd5           = sum(m['gd']           for m in o5)
        opp_avg_goals5    = float(np.mean([m['goals_for']     for m in o5]))
        opp_avg_conceded5 = float(np.mean([m['goals_against'] for m in o5]))
        opp_wr10          = float(np.mean([m['win']           for m in o10]))

    # Head-to-head stats
    h2h      = list(h2h_history[(team, opp)])
    has_h2h  = len(h2h) > 0
    h2h_pts5 = sum(m['points'] for m in h2h) if has_h2h else 0
    h2h_gd5  = sum(m['gd']     for m in h2h) if has_h2h else 0

    return {
        'days_since_last_match_team': dslm_team,
        'days_since_last_match_opp':  dslm_opp,
        'first_match_team':           first_team,
        'first_match_opp':            first_opp,
        'team_points_last5':          team_pts5,
        'team_points_last10':         team_pts10,
        'team_gd_last5':              team_gd5,
        'team_avg_goals_last5':       team_avg_goals5,
        'team_avg_conceded_last5':    team_avg_conceded5,
        'team_win_rate_last10':       team_wr10,
        'opp_points_last5':           opp_pts5,
        'opp_points_last10':          opp_pts10,
        'opp_gd_last5':               opp_gd5,
        'opp_avg_goals_last5':        opp_avg_goals5,
        'opp_avg_conceded_last5':     opp_avg_conceded5,
        'opp_win_rate_last10':        opp_wr10,
        'points_last5_diff':          team_pts5 - opp_pts5,
        'gd_last5_diff':              team_gd5  - opp_gd5,
        'h2h_points_last5':           h2h_pts5,
        'h2h_gd_last5':               h2h_gd5,
        '_h2h_has_data':              has_h2h,   # internal flag, not a model feature
    }


def compute_dynamic_fe(ts):
    """
    Compute FE columns that are DERIVED from time-series features,
    mirroring feature_engineering.ipynb steps 6-10.
    Static FE columns (ELO, rank, env, tournament_tier …) are taken
    directly from df_test_fe and NOT recomputed here.
    """
    fe = {}

    # Form & Momentum (step 6)
    fe['attack_vs_def_team']      = ts['team_avg_goals_last5'] - ts['opp_avg_conceded_last5']
    fe['attack_vs_def_opp']       = ts['opp_avg_goals_last5']  - ts['team_avg_conceded_last5']
    fe['xg_advantage']            = fe['attack_vs_def_team'] - fe['attack_vs_def_opp']
    fe['momentum_diff']           = ts['team_win_rate_last10'] - ts['opp_win_rate_last10']
    fe['momentum_ratio']          = ts['team_win_rate_last10'] / (ts['opp_win_rate_last10'] + 1e-5)
    fe['pts_diff_last10']         = ts['team_points_last10']   - ts['opp_points_last10']
    fe['team_attack_consistency'] = ts['team_avg_goals_last5'] / (ts['team_avg_conceded_last5'] + 1e-5)
    fe['opp_attack_consistency']  = ts['opp_avg_goals_last5']  / (ts['opp_avg_conceded_last5']  + 1e-5)
    fe['attack_consistency_diff'] = fe['team_attack_consistency'] - fe['opp_attack_consistency']

    # H2H (step 7)
    fe['h2h_dominance']    = ts['h2h_points_last5'] / 15
    fe['h2h_gd_per_match'] = ts['h2h_gd_last5']    / 5
    fe['h2h_missing']      = int(not ts['_h2h_has_data'])

    # Rest / Fatigue (step 10)
    def rest_cat(d):
        if d <= 4:  return 0   # tight turnaround (or first match → d=0)
        if d <= 10: return 1   # normal rest
        return 2               # long rest / international break

    d_t = ts['days_since_last_match_team']
    d_o = ts['days_since_last_match_opp']
    fe['rest_diff']      = d_t - d_o
    fe['rest_cat_team']  = rest_cat(d_t)
    fe['rest_cat_opp']   = rest_cat(d_o)
    fe['rest_advantage'] = fe['rest_cat_team'] - fe['rest_cat_opp']

    return fe

Helper functions defined ✓


## 6. Sequential Prediction Loop

In [29]:
predictions = []   # list of {'Id': ..., 'team_goals': ..., 'opp_goals': ...}

match_ids_ordered = df_test_fe['match_id'].unique()   # already sorted ascending

for mid in match_ids_ordered:
    # Get the two rows for this physical match
    pair = df_test_fe[df_test_fe['match_id'] == mid]

    if len(pair) < 2:
        print(f"[WARN] match_id={mid} has {len(pair)} row(s) — skipping.")
        continue

    first_row  = pair.iloc[0]
    second_row = pair.iloc[1]

    team       = first_row['team']
    opp        = first_row['opponent']
    match_date = first_row['date']

    # Recompute time-series and derived FE features
    ts     = compute_ts_features(team, opp, match_date)
    fe_dyn = compute_dynamic_fe(ts)

    # Build feature dict: static values from df_test_fe + dynamic overrides
    row_dict = first_row.to_dict()
    row_dict.update(ts)        # overwrite TS columns with freshly computed values
    row_dict.update(fe_dyn)    # overwrite derived FE columns

    # Assemble X in the exact column order the model was trained with
    X_pred = pd.DataFrame([{col: row_dict.get(col, np.nan) for col in FEATURE_COLS}])

    for col in X_pred.columns:
        # If the column is text/categorical, fill with 'Unknown'
        if X_pred[col].dtype == 'object' or X_pred[col].dtype.name == 'category':
            X_pred[col] = X_pred[col].fillna('Unknown')
        # If the column is numeric, fill with -999 (The Tree Anomaly trick)
        else:
            X_pred[col] = X_pred[col].fillna(-999)

    # Predict
    raw      = model.predict(X_pred)[0]            # [team_goals_pred, opp_goals_pred]
    t_goals  = int(max(0, round(float(raw[0]))))
    o_goals  = int(max(0, round(float(raw[1]))))

    # Store predictions
    # Row 1 (team perspective) — predicted directly
    predictions.append({'Id': first_row['Id'],
                        'team_goals': t_goals, 'opp_goals': o_goals})
    # Row 2 (opponent perspective) — mirror: swap team_goals ↔ opp_goals
    predictions.append({'Id': second_row['Id'],
                        'team_goals': o_goals, 'opp_goals': t_goals})

    # Update history with this result (for all subsequent matches)
    update_history(team, opp, t_goals, o_goals, match_date)

print(f"\nPredictions generated: {len(predictions)} rows "
      f"({len(predictions)//2} matches)")


Predictions generated: 42422 rows (21211 matches)


## 7. Assemble & Validate Submission

In [30]:
ss         = pd.read_csv(PATH_OUTPUT)
pred_df    = pd.DataFrame(predictions)
submission = ss[['Id']].merge(pred_df, on='Id', how='left')

# Sanity checks
n_missing = submission[['team_goals', 'opp_goals']].isna().sum().sum()
if n_missing > 0:
    print(f"[WARN] {n_missing} cell(s) still NaN — Id mismatch between "
          f"df_test_fe and sample submission?")
else:
    print("All rows filled — no NaNs detected.")

print(f"\nShape: {submission.shape}")
display(submission.head(20))

print("\n── team_goals distribution ──")
print(submission['team_goals'].value_counts().sort_index())
print("\n── opp_goals distribution ──")
print(submission['opp_goals'].value_counts().sort_index())

All rows filled — no NaNs detected.

Shape: (42422, 3)


,Id,team_goals,opp_goals
0,M034984_Seychelles,1,5
1,M034984_Mauritius,5,1
2,M034985_Comoros,1,5
3,M034985_Maldives,5,1
4,M034986_Réunion,1,5
5,M034986_Madagascar,5,1
6,M034987_El Salvador,5,1
7,M034987_Venezuela,1,5
8,M034988_Mayotte,5,3
9,M034988_Réunion,3,5



── team_goals distribution ──
team_goals
0      1272
1     15574
2      3318
3      1671
4      6877
5      8207
6      3574
7      1321
8       389
9       166
10       44
11        8
12        1
Name: count, dtype: int64

── opp_goals distribution ──
opp_goals
0      1272
1     15574
2      3318
3      1671
4      6877
5      8207
6      3574
7      1321
8       389
9       166
10       44
11        8
12        1
Name: count, dtype: int64


## 8. Export Submission

In [31]:
submission.to_csv("C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Submission/gb_submission.csv", index=False)
display(submission.head(5))

,Id,team_goals,opp_goals
0,M034984_Seychelles,1,5
1,M034984_Mauritius,5,1
2,M034985_Comoros,1,5
3,M034985_Maldives,5,1
4,M034986_Réunion,1,5
